In [1]:
import logging
import os
import random
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import scipy.stats as stat

from utils import (
    load_splits,
    build_dataset_from_ids,
    compute_bounding_box, bbox_to_padded_shape,
    load_brain_mask,
    load_preprocessing_stats
)

from evaluate import (
    _apply_norm, _denormalize,
    subject_metrics, evaluate_model, summarize
)

from compare import _print_table
from config import Config
from pathlib import Path
import json

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# Experiment setup
- for each latent dimension:
- do k-fold stratified (so that we get representative groups for survival - balanced censoring etc), repeated (?) cross validation for reporting robust results
- in each fold - train the DeepSurv network (simple MLP?) - to potentially get predictive power from the latent codes that's better than the Cox itself - maybe better than for the PCs?
- or potentially other models from this toolkit?
- BUT whether the predictions are "better" would be the question IF we cheked the PCs with the same model (the DeepSurv) and then the reasosning could be made - otherwise we're comparing the c-index on Cox for PCs vs c-index using DeepSurv

- so what this experiment cheks is whether ANYTHING can be taken out of the latent components (bcs with Cox it couldn't) - potentially due to the collinearity of the Latent comps (visible in the hazard ratios plots)


# Load latent codes

In [2]:
from cox import (_get_or_encode, _build_df)
from utils import unpad_ucsf_ids

In [3]:
LATENT_DIMS      = [2, 4, 6, 8, 12, 16, 32, 64, 128]
DURATION_COL     = "OS (days) - corrected"
EVENT_COL        = "status"
CLINICAL_COLS    = ["age", "sex"]
N_SPLITS         = 5

In [4]:
base_cfg = Config()
splits = load_splits(base_cfg.splits_dir)
train_val_ids = splits["train"] + splits["val"]
test_ids      = splits["test"]
all_ids       = train_val_ids + test_ids

In [5]:
len(all_ids)

1001

In [6]:
DIM = 4
with open(Path(base_cfg.jsons_dir) / "id_to_cohort.json") as f:
    id_to_cohort = json.load(f)

clinical = pd.read_csv(base_cfg.clinical_csv)
out_dir = Path(base_cfg.jsons_dir)

run_cfg = Config()
run_cfg.latent_dim       = DIM
run_cfg.normalisation    = "zscore"
run_cfg.use_lr_scheduler = True
device = run_cfg.device
print(device)

cuda


In [7]:
# calculate stats on train only
stats = load_preprocessing_stats(str(out_dir / "preprocessing_stats.json"))

In [8]:
codes, ids = _get_or_encode(run_cfg, all_ids, id_to_cohort, stats, device)
df, z_cols = _build_df(codes, ids, clinical, DIM)

2026-08-31 15:37:31,151 INFO Loading cached codes from /home/joan/Desktop/PROJECTS/Julia/code/dl-tract-density-survival/latents/latent4_zscore_cosinelr


In [9]:
df

,z1,z2,z3,z4,id,ID,age,sex,eor,mgmt,...,Core lesion TDMap,Non-enhancing TDMap,Non-enhancing lesion TDMap,Enhancing TDMap,Enhancing lesion TDMap,Core+Enhancing TDMap,Core+Enhancing lesion TDMap,cohort,site,OS (days) - corrected
0,1.621464,-5.693500,35.877537,-4.840298,UPENN-GBM-00147_11,UPENN-GBM-00147_11,82.41,1,1.0,0.0,...,0.804185,130.299588,2.866155,74.560762,1.046598,100.428056,1.047600,1,1,107.143828
1,12.845091,-1.782956,26.826283,-24.796284,UCSF-PDGM-430,UCSF-PDGM-430,79.00,0,2.0,0.0,...,0.214835,255.461534,2.799048,133.118295,0.736955,123.039131,0.736956,0,0,406.000000
2,21.200544,18.810493,4.611731,-28.574509,UCSF-PDGM-472,UCSF-PDGM-472,76.00,1,0.0,2.0,...,1.278647,124.437120,4.896154,117.381894,2.580636,123.730019,2.580733,0,0,38.000000
3,13.924551,-10.392248,24.448484,-0.469693,UCSF-PDGM-332,UCSF-PDGM-332,68.00,0,2.0,2.0,...,0.736385,175.369296,5.865094,146.804520,1.725475,148.421375,1.725487,0,0,42.000000
4,0.680297,-6.521086,6.852678,5.157828,UPENN-GBM-00337_11,UPENN-GBM-00337_11,62.34,0,2.0,NaN,...,2.775009,112.819121,4.693732,157.227826,3.901126,166.981916,3.917241,1,1,515.629674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,-8.085268,22.565022,0.704565,-9.653357,TCGA-02-0009,TCGA-02-0009,61.00,1,NaN,0.0,...,0.379249,70.014250,1.257232,148.231487,0.614384,88.865209,0.650983,2,1,430.967790
995,16.352362,-11.631674,23.623861,-10.405519,TCGA-06-0238,TCGA-06-0238,46.00,0,NaN,NaN,...,0.466644,99.463595,3.751923,56.781503,0.673405,54.285974,0.678839,2,1,542.055761
996,17.940706,14.994812,-19.931723,-23.902170,TCGA-76-6664,TCGA-76-6664,49.00,1,NaN,2.0,...,0.462849,93.481570,3.777519,46.345565,0.955934,45.520411,1.009172,2,1,317.203001
997,8.373868,-16.543245,24.541079,6.661209,TCGA-06-0241,TCGA-06-0241,65.00,1,NaN,NaN,...,0.780565,171.162536,1.740584,97.543244,1.220562,107.294998,1.223833,2,1,608.976225


In [10]:
# split the whole cohort into train / val / test
from sklearn.model_selection import train_test_split

# Combined stratification key: e.g. "ucsf_censored", "upenn_uncensored", etc.
df["strat_key"] = df["cohort"].astype(str) + "_" + df["status"].astype(str)

print("Stratification group sizes:")
print(df["strat_key"].value_counts().sort_index())

TEST_SIZE = 0.2
VAL_SIZE  = 0.2  # fraction of the remaining (non-test) pool -> ~16% of total

trainval_idx, test_idx = train_test_split(
    df.index,  # RangeIndex object
    test_size=TEST_SIZE,
    random_state=420,
    shuffle=True,
    stratify=df["strat_key"]
)
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=VAL_SIZE,
    random_state=420,
    shuffle=True,
    stratify=df.loc[trainval_idx, "strat_key"]
)

df_train = df.loc[train_idx].copy()
df_val   = df.loc[val_idx].copy()
df_test  = df.loc[test_idx].copy()

# Verify
for split_name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    print(f"\n{split_name} (N={len(split_df)}):")
    print("  Cohort proportions:\n", split_df["cohort"].value_counts(normalize=True).round(3))
    print("  Censoring rate:    ", (split_df["status"] == 0).mean().round(3))

Stratification group sizes:
strat_key
0_0.0    144
0_1.0    223
1_0.0     14
1_1.0    482
2_0.0     21
2_1.0     82
3_0.0      9
3_1.0     24
Name: count, dtype: int64

Train (N=639):
  Cohort proportions:
 cohort
1    0.498
0    0.366
2    0.103
3    0.033
Name: proportion, dtype: float64
  Censoring rate:     0.188

Val (N=160):
  Cohort proportions:
 cohort
1    0.494
0    0.369
2    0.106
3    0.031
Name: proportion, dtype: float64
  Censoring rate:     0.188

Test (N=200):
  Cohort proportions:
 cohort
1    0.495
0    0.370
2    0.100
3    0.035
Name: proportion, dtype: float64
  Censoring rate:     0.19


In [11]:
# Standardize latent codes + age (fit on train only, to avoid leakage)
from sklearn.preprocessing import StandardScaler

SCALE_COLS = z_cols + ["age"]

scaler = StandardScaler()
df_train[SCALE_COLS] = scaler.fit_transform(df_train[SCALE_COLS])
df_val[SCALE_COLS]   = scaler.transform(df_val[SCALE_COLS])
df_test[SCALE_COLS]  = scaler.transform(df_test[SCALE_COLS])

# Build & fit DeepSurv (0 hidden layers = linear Cox sanity check)

In [12]:
import torchtuples as tt
from pycox.models import CoxPH
import torch.nn as nn

FEATURE_COLS = z_cols + ["age", "sex"]

x_train = df_train[FEATURE_COLS].values.astype("float32")
x_val   = df_val[FEATURE_COLS].values.astype("float32")
x_test  = df_test[FEATURE_COLS].values.astype("float32")

get_target = lambda d: (d[DURATION_COL].values.astype("float32"),
                         d[EVENT_COL].values.astype("float32"))
y_train = get_target(df_train)
y_val   = get_target(df_val)
durations_test, events_test = get_target(df_test)

net = tt.practical.MLPVanilla(
    in_features=len(FEATURE_COLS),
    num_nodes=[],          # no hidden layers -> linear Cox, sanity check first
    out_features=1,
    batch_norm=True,
    dropout=0.1,
    output_bias=False,
    activation=nn.ReLU
)

model = CoxPH(net, tt.optim.Adam)
model.optimizer.set_lr(0.01)

log = model.fit(
    x_train, y_train,
    batch_size=256, epochs=100,
    val_data=(x_val, y_val),
    val_batch_size=256,
    callbacks=[tt.callbacks.EarlyStopping()],
)

0:	[0s / 0s],		train_loss: 4.5406,	val_loss: 4.2384
1:	[0s / 0s],		train_loss: 4.5107,	val_loss: 4.2084
2:	[0s / 0s],		train_loss: 4.4536,	val_loss: 4.1826
3:	[0s / 0s],		train_loss: 4.4175,	val_loss: 4.1613
4:	[0s / 0s],		train_loss: 4.4073,	val_loss: 4.1442
5:	[0s / 0s],		train_loss: 4.3692,	val_loss: 4.1309
6:	[0s / 0s],		train_loss: 4.3525,	val_loss: 4.1212
7:	[0s / 0s],		train_loss: 4.3366,	val_loss: 4.1147
8:	[0s / 0s],		train_loss: 4.3246,	val_loss: 4.1106
9:	[0s / 0s],		train_loss: 4.3111,	val_loss: 4.1086
10:	[0s / 0s],		train_loss: 4.3092,	val_loss: 4.1080
11:	[0s / 0s],		train_loss: 4.2920,	val_loss: 4.1083
12:	[0s / 0s],		train_loss: 4.3058,	val_loss: 4.1095
13:	[0s / 0s],		train_loss: 4.2989,	val_loss: 4.1112
14:	[0s / 0s],		train_loss: 4.2970,	val_loss: 4.1130
15:	[0s / 0s],		train_loss: 4.3016,	val_loss: 4.1143
16:	[0s / 0s],		train_loss: 4.2808,	val_loss: 4.1150
17:	[0s / 0s],		train_loss: 4.2982,	val_loss: 4.1157
18:	[0s / 0s],		train_loss: 4.2885,	val_loss: 4.1158
19:

In [13]:
_ = model.compute_baseline_hazards()
surv = model.predict_surv_df(x_test)

from pycox.evaluation import EvalSurv
ev = EvalSurv(surv, durations_test, events_test, censor_surv="km")
print("pycox CoxPH (0 hidden layers) concordance:", ev.concordance_td())

pycox CoxPH (0 hidden layers) concordance: 0.6134448687640177


In [14]:
# Sanity check: a 0-hidden-layer pycox CoxPH should behave like a linear Cox model
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

cph = CoxPHFitter()
cph.fit(
    df_train[FEATURE_COLS + [DURATION_COL, EVENT_COL]],
    duration_col=DURATION_COL,
    event_col=EVENT_COL,
)

lifelines_risk = cph.predict_partial_hazard(df_test[FEATURE_COLS])
lifelines_cindex = concordance_index(durations_test, -lifelines_risk, events_test)
print("lifelines CoxPHFitter concordance:   ", lifelines_cindex)

lifelines CoxPHFitter concordance:    0.6329813061422676


# Clinical-only baseline (age + sex)

In [15]:
FEATURE_COLS_CLINICAL = ["age", "sex"]

x_train_clin = df_train[FEATURE_COLS_CLINICAL].values.astype("float32")
x_val_clin   = df_val[FEATURE_COLS_CLINICAL].values.astype("float32")
x_test_clin  = df_test[FEATURE_COLS_CLINICAL].values.astype("float32")

net_clin = tt.practical.MLPVanilla(
    in_features=len(FEATURE_COLS_CLINICAL),
    num_nodes=[],          # linear baseline; bump to match the latent-code net's hidden layers for an apples-to-apples nonlinear comparison
    out_features=1,
    batch_norm=False,
    dropout=None,
    output_bias=False,
)

model_clin = CoxPH(net_clin, tt.optim.Adam)
model_clin.optimizer.set_lr(0.01)

log_clin = model_clin.fit(
    x_train_clin, y_train,
    batch_size=256, epochs=100,
    val_data=(x_val_clin, y_val),
    val_batch_size=256,
    callbacks=[tt.callbacks.EarlyStopping()],
)

0:	[0s / 0s],		train_loss: 4.2940,	val_loss: 4.0916
1:	[0s / 0s],		train_loss: 4.3048,	val_loss: 4.0922
2:	[0s / 0s],		train_loss: 4.2972,	val_loss: 4.0928
3:	[0s / 0s],		train_loss: 4.3067,	val_loss: 4.0934
4:	[0s / 0s],		train_loss: 4.3021,	val_loss: 4.0935
5:	[0s / 0s],		train_loss: 4.2867,	val_loss: 4.0937
6:	[0s / 0s],		train_loss: 4.2921,	val_loss: 4.0936
7:	[0s / 0s],		train_loss: 4.3037,	val_loss: 4.0935
8:	[0s / 0s],		train_loss: 4.2878,	val_loss: 4.0937
9:	[0s / 0s],		train_loss: 4.2994,	val_loss: 4.0940
10:	[0s / 0s],		train_loss: 4.2970,	val_loss: 4.0941


In [16]:
_ = model_clin.compute_baseline_hazards()
surv_clin = model_clin.predict_surv_df(x_test_clin)

ev_clin = EvalSurv(surv_clin, durations_test, events_test, censor_surv="km")
print("pycox CoxPH (age + sex only) concordance:", ev_clin.concordance_td())

pycox CoxPH (age + sex only) concordance: 0.6375704673577014


# Hyperparameter search (Optuna) for DeepSurv

In [17]:
import optuna
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)


def build_net(in_features, num_nodes, activation_name, dropout):
    if not num_nodes:
        return tt.practical.MLPVanilla(
            in_features=in_features, num_nodes=[], out_features=1,
            batch_norm=False, dropout=None, output_bias=False,
        )
    activation_cls = {"relu": nn.ReLU, "selu": nn.SELU}[activation_name]
    # SELU expects LeCun-normal init; MLPVanilla defaults to Kaiming/ReLU-gain otherwise
    w_init_ = (lambda w: nn.init.kaiming_normal_(w, nonlinearity="linear")) if activation_name == "selu" \
              else (lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"))
    return tt.practical.MLPVanilla(
        in_features=in_features, num_nodes=num_nodes, out_features=1,
        batch_norm=False, dropout=dropout if dropout > 0 else None,
        activation=activation_cls, output_bias=False, w_init_=w_init_,
    )


def build_optimizer(name, lr, weight_decay, momentum=None):
    if name == "adam":
        return tt.optim.Adam(lr=lr, weight_decay=weight_decay)
    return tt.optim.SGD(lr=lr, weight_decay=weight_decay, momentum=momentum)


def evaluate_config(hyperparams, splitter, df_all=df, scale_cols=SCALE_COLS,
                     feature_cols=FEATURE_COLS, val_size=0.2, epochs=100, seed=42):
    """K-fold evaluator: each fold's held-out chunk is scored; the remaining
    folds are further split into an inner train/val used only for early
    stopping (never scored). Returns one concordance score per fold."""
    scores = []
    for fold_train_pos, fold_test_pos in splitter.split(np.zeros(len(df_all)), df_all["strat_key"]):
        fold_train_full = df_all.iloc[fold_train_pos]
        fold_test = df_all.iloc[fold_test_pos].copy()

        inner_train_idx, inner_val_idx = train_test_split(
            fold_train_full.index, test_size=val_size, random_state=seed,
            shuffle=True, stratify=fold_train_full["strat_key"],
        )
        inner_train = fold_train_full.loc[inner_train_idx].copy()
        inner_val = fold_train_full.loc[inner_val_idx].copy()

        fold_scaler = StandardScaler()
        inner_train[scale_cols] = fold_scaler.fit_transform(inner_train[scale_cols])
        inner_val[scale_cols] = fold_scaler.transform(inner_val[scale_cols])
        fold_test[scale_cols] = fold_scaler.transform(fold_test[scale_cols])

        x_inner_train = inner_train[feature_cols].values.astype("float32")
        x_inner_val = inner_val[feature_cols].values.astype("float32")
        x_fold_test = fold_test[feature_cols].values.astype("float32")
        y_inner_train = get_target(inner_train)
        y_inner_val = get_target(inner_val)
        y_fold_test = get_target(fold_test)

        net = build_net(len(feature_cols), hyperparams["num_nodes"], hyperparams["activation"], hyperparams["dropout"])
        optimizer = build_optimizer(hyperparams["optimizer"], hyperparams["lr"],
                                     hyperparams["weight_decay"], hyperparams.get("momentum"))
        fold_model = CoxPH(net, optimizer)

        try:
            fold_model.fit(
                x_inner_train, y_inner_train,
                batch_size=hyperparams["batch_size"], epochs=epochs,
                val_data=(x_inner_val, y_inner_val), val_batch_size=256,
                callbacks=[tt.callbacks.EarlyStopping()], verbose=False,
            )
            _ = fold_model.compute_baseline_hazards()
            surv_fold_test = fold_model.predict_surv_df(x_fold_test)
            ev_fold = EvalSurv(surv_fold_test, *y_fold_test, censor_surv="km")
            score = ev_fold.concordance_td()
        except Exception:
            score = 0.5  # chance level, for diverged/failed configs
        scores.append(score if np.isfinite(score) else 0.5)

    return np.array(scores)


def evaluate_lifelines(splitter, df_all=df, feature_cols=FEATURE_COLS, scale_cols=SCALE_COLS):
    """Same fold loop as evaluate_config, but for the (non-neural) lifelines
    baseline -- no inner val split needed since there's no early stopping."""
    scores = []
    for fold_train_pos, fold_test_pos in splitter.split(np.zeros(len(df_all)), df_all["strat_key"]):
        fold_train = df_all.iloc[fold_train_pos].copy()
        fold_test = df_all.iloc[fold_test_pos].copy()

        fold_scaler = StandardScaler()
        fold_train[scale_cols] = fold_scaler.fit_transform(fold_train[scale_cols])
        fold_test[scale_cols] = fold_scaler.transform(fold_test[scale_cols])

        cph_fold = CoxPHFitter()
        cph_fold.fit(fold_train[feature_cols + [DURATION_COL, EVENT_COL]],
                      duration_col=DURATION_COL, event_col=EVENT_COL)
        risk = cph_fold.predict_partial_hazard(fold_test[feature_cols])
        scores.append(concordance_index(fold_test[DURATION_COL], -risk, fold_test[EVENT_COL]))

    return np.array(scores)


def ci95(scores):
    mean = scores.mean()
    sem = scores.std(ddof=1) / np.sqrt(len(scores))
    lo, hi = stat.t.interval(0.95, df=len(scores) - 1, loc=mean, scale=sem)
    return mean, lo, hi


def sample_hyperparams(trial):
    n_layers = trial.suggest_int("n_layers", 0, 2)
    num_nodes = [trial.suggest_categorical(f"width_{i}", [4, 8, 16, 32]) for i in range(n_layers)]
    activation = trial.suggest_categorical("activation", ["relu", "selu"])
    dropout = trial.suggest_categorical("dropout", [0.0, 0.1, 0.2, 0.3, 0.4, 0.5])
    optimizer = trial.suggest_categorical("optimizer", ["adam", "sgd"])
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    momentum = trial.suggest_float("momentum", 0.0, 0.99) if optimizer == "sgd" else None
    return dict(num_nodes=num_nodes, activation=activation, dropout=dropout,
                optimizer=optimizer, lr=lr, weight_decay=weight_decay,
                batch_size=batch_size, momentum=momentum)

In [18]:
N_TRIALS = 25


def objective(trial):
    hp = sample_hyperparams(trial)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    scores = evaluate_config(hp, skf)
    return float(scores.mean())


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best mean {N_SPLITS}-fold CV concordance:", study.best_value)
print("Best params:", study.best_params)

  0%|          | 0/25 [00:00<?, ?it/s]

Best mean 5-fold CV concordance: 0.6124730907084492
Best params: {'n_layers': 0, 'activation': 'selu', 'dropout': 0.5, 'optimizer': 'adam', 'lr': 0.0148841033144882, 'weight_decay': 0.00012491478808383512, 'batch_size': 64}


In [19]:
trials_df = study.trials_dataframe().sort_values("value", ascending=False)
trials_df

,number,value,datetime_start,datetime_complete,duration,params_activation,params_batch_size,params_dropout,params_lr,params_momentum,params_n_layers,params_optimizer,params_weight_decay,params_width_0,params_width_1,state
22,22,0.612473,2026-08-31 15:38:05.409821,2026-08-31 15:38:06.145144,0 days 00:00:00.735323,selu,64,0.5,0.014884,NaN,0,adam,0.000125,NaN,NaN,COMPLETE
5,5,0.611970,2026-08-31 15:37:43.618892,2026-08-31 15:37:46.411117,0 days 00:00:02.792225,selu,64,0.1,0.004150,0.225656,0,sgd,0.010894,NaN,NaN,COMPLETE
23,23,0.610775,2026-08-31 15:38:06.145814,2026-08-31 15:38:06.963317,0 days 00:00:00.817503,selu,64,0.1,0.012583,NaN,0,adam,0.000103,NaN,NaN,COMPLETE
17,17,0.610680,2026-08-31 15:38:01.853059,2026-08-31 15:38:02.511106,0 days 00:00:00.658047,selu,64,0.1,0.027404,NaN,0,adam,0.000199,NaN,NaN,COMPLETE
11,11,0.609707,2026-08-31 15:37:52.992548,2026-08-31 15:37:54.301626,0 days 00:00:01.309078,selu,64,0.5,0.004235,NaN,0,adam,0.000002,NaN,NaN,COMPLETE
12,12,0.607906,2026-08-31 15:37:54.302274,2026-08-31 15:37:55.336803,0 days 00:00:01.034529,selu,64,0.5,0.010416,NaN,0,adam,0.001797,NaN,NaN,COMPLETE
24,24,0.607862,2026-08-31 15:38:06.963812,2026-08-31 15:38:07.773017,0 days 00:00:00.809205,selu,64,0.0,0.013028,NaN,0,adam,0.000072,NaN,NaN,COMPLETE
10,10,0.607392,2026-08-31 15:37:51.754118,2026-08-31 15:37:52.991898,0 days 00:00:01.237780,selu,64,0.5,0.003324,NaN,0,adam,0.000002,NaN,NaN,COMPLETE
19,19,0.607209,2026-08-31 15:38:03.059205,2026-08-31 15:38:03.451418,0 days 00:00:00.392213,selu,128,0.1,0.019234,NaN,0,adam,0.096494,NaN,NaN,COMPLETE
18,18,0.606763,2026-08-31 15:38:02.511748,2026-08-31 15:38:03.058605,0 days 00:00:00.546857,selu,256,0.1,0.050193,0.426148,0,sgd,0.000203,NaN,NaN,COMPLETE


In [20]:
import optuna.visualization as vis

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

## Final evaluation of the winning config: repeated 6×5-fold CV vs. baselines (same folds)

In [21]:
best = study.best_params
best_hp = dict(
    num_nodes=[best[f"width_{i}"] for i in range(best["n_layers"])],
    activation=best["activation"],
    dropout=best["dropout"],
    optimizer=best["optimizer"],
    lr=best["lr"],
    weight_decay=best["weight_decay"],
    batch_size=best["batch_size"],
    momentum=best.get("momentum"),
)

clinical_hp = dict(
    num_nodes=[], activation="relu", dropout=0.0,
    optimizer="adam", lr=0.01, weight_decay=0.0, batch_size=256, momentum=None,
)

rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=6, random_state=42)

deepsurv_scores = evaluate_config(best_hp, rskf)
clinical_scores = evaluate_config(clinical_hp, rskf, scale_cols=["age"], feature_cols=FEATURE_COLS_CLINICAL)
lifelines_scores = evaluate_lifelines(rskf)

for name, scores in [
    (f"Tuned DeepSurv (DIM={DIM})", deepsurv_scores),
    ("Clinical-only net (age+sex)", clinical_scores),
    ("lifelines CoxPHFitter", lifelines_scores),
]:
    mean, lo, hi = ci95(scores)
    print(f"{name:<30} {mean:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  (n_folds={len(scores)})")

Tuned DeepSurv (DIM=4)         0.6071  95% CI [0.5989, 0.6152]  (n_folds=30)
Clinical-only net (age+sex)    0.6123  95% CI [0.6042, 0.6204]  (n_folds=30)
lifelines CoxPHFitter          0.6084  95% CI [0.6004, 0.6163]  (n_folds=30)
